# Data Preprocessing Pipeline - Categorical & Numerical

## Proper pipeline with NO data leakage

**Key Principle:** Fit all transformers (scaler, encoders) on TRAIN data only, then apply to VAL and TEST

**Output:** 3 clean CSV files (train.csv, validation.csv, test.csv)

---

## 1. Setup & Configuration

In [ ]:
import sys
import os
sys.path.append(os.path.abspath("../configs"))
import pickle
import warnings
from read_config import get_config
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")

print("Libraries imported")

In [ ]:
file_path = get_config("data_path")

In [ ]:
# === CONFIGURATION ===

INPUT_CSV = "fifa_world_cup_2026_player_performance.csv"
OUTPUT_DIR = "../data/processed"

# Create output directory
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Split ratios
TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
TEST_RATIO = 0.15
RANDOM_SEED = 42
REMOVE_ZERO_RATINGS = True  # Remove non-playing appearances

print("   Configuration:")
print(f"  Input: {file_path}")
print(f"  Output: {OUTPUT_DIR}")
print(f"  Split: {TRAIN_RATIO}/{VAL_RATIO}/{TEST_RATIO} (train/val/test)")
print(f"  Remove zeros: {REMOVE_ZERO_RATINGS}")

## 2. Define Features

In [ ]:
# === FEATURE DEFINITIONS (OPTION 3 - COMPREHENSIVE) ===

NUMERIC_FEATURES = [
    # Tier 1: Primary Predictors
    "consistency_score",
    "market_value_eur",
    "pressure_resistance",
    "defensive_contribution",
    "clutch_performance_score",
    "offensive_contribution",
    "creativity_score",
    "stamina_score",
    # Tier 2: Demographics
    "age",
    "height_cm",
    "weight_kg",
    "minutes_played",
    # Tier 3: Actions
    "goals",
    "tackles",
    "interceptions",
    "key_passes",
    "expected_goals_xg",
    "expected_assists_xa",
    "defensive_actions",
    "recoveries",
    "assists",
    # Tier 4: Physical
    "top_speed_kmh",
    "distance_covered_km",
    "accelerations",
    "decelerations",
    "sprint_distance_km",
]

CATEGORICAL_FEATURES = [
    "position",
    "tournament_stage",
    "match_result",
]

TARGET = "player_rating"

print("  Feature Definition:")
print(f"  Numeric: {len(NUMERIC_FEATURES)} features")
print(f"  Categorical: {len(CATEGORICAL_FEATURES)} features")
print(f"  Target: {TARGET}")

## 3. Load & Explore Data

In [ ]:
# Load data
df = pd.read_csv(file_path)

print("\n Original Data")
print(f"  Shape: {df.shape}")
print(f"  Rows: {df.shape[0]:,}")
print("  Target stats:")
print(f"    Mean: {df[TARGET].mean():.2f}")
print(f"    Std: {df[TARGET].std():.2f}")
print(f"    Min: {df[TARGET].min():.2f}")
print(f"    Max: {df[TARGET].max():.2f}")
print(f"    Non-zero: {(df[TARGET] > 0).sum():,}")

In [ ]:
# Remove non-playing appearances
if REMOVE_ZERO_RATINGS:
    print("\n Filtering")
    print(f"  Before: {len(df):,} rows")
    df = df[df[TARGET] > 0].reset_index(drop=True)
    print(f"  After: {len(df):,} rows")
    print(f"  Removed: {len(pd.read_csv(INPUT_CSV)) - len(df):,} zero ratings")

print("\nReady for preprocessing")

## 4. Handle Missing Values

In [ ]:
print("\n Checking Missing Values")

# Numeric features
missing_numeric = df[NUMERIC_FEATURES].isnull().sum()
if missing_numeric.sum() > 0:
    print("\nNumeric missing:")
    print(missing_numeric[missing_numeric > 0])
else:
    print(" No missing numeric values")

# Categorical features
missing_cat = df[CATEGORICAL_FEATURES].isnull().sum()
if missing_cat.sum() > 0:
    print("\nCategorical missing:")
    print(missing_cat[missing_cat > 0])
else:
    print(" No missing categorical values")

## 5. Feature Engineering (Before Split)

In [ ]:
print("\n  Feature Engineering")

# Create efficiency metrics
df["pass_accuracy"] = (df["successful_passes"] / (df["total_passes"] + 1)) * 100
df["shot_accuracy"] = (df["shots_on_target"] / (df["shots"] + 1)) * 100
df["dribble_success"] = (df["successful_dribbles"] / (df["dribbles_attempted"] + 1)) * 100
df["goals_per_minute"] = (df["goals"] / (df["minutes_played"] + 1)) * 90
df["distance_per_minute"] = df["distance_covered_km"] / (df["minutes_played"] + 1)

ENGINEERED_FEATURES = ["pass_accuracy", "shot_accuracy", "dribble_success", "goals_per_minute", "distance_per_minute"]
NUMERIC_FEATURES.extend(ENGINEERED_FEATURES)

print(f"  Created: {len(ENGINEERED_FEATURES)} engineered features")
print(f"  Total numeric: {len(NUMERIC_FEATURES)}")

## 6. Train-Validation-Test Split (FIRST - Before Preprocessing!)

In [ ]:
print("\n  SPLITTING DATA (Before any preprocessing!)")
print("=" * 70)
print("This ensures NO data leakage when fitting transformers")

# Extract features and target
X = df[NUMERIC_FEATURES + CATEGORICAL_FEATURES].copy()
y = df[TARGET].copy()

# Split 1: Train vs Temp (Val+Test)
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=(VAL_RATIO + TEST_RATIO), random_state=RANDOM_SEED, shuffle=True
)

# Split 2: Val vs Test
val_test_split = TEST_RATIO / (VAL_RATIO + TEST_RATIO)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=val_test_split, random_state=RANDOM_SEED, shuffle=True
)

print("\n Split Results:")
print(f"  Train: {len(X_train):6,} samples ({len(X_train) / len(X) * 100:.1f}%)")
print(f"  Val:   {len(X_val):6,} samples ({len(X_val) / len(X) * 100:.1f}%)")
print(f"  Test:  {len(X_test):6,} samples ({len(X_test) / len(X) * 100:.1f}%)")
print(f"  Total: {len(X):6,} samples")

print("\n Split complete - No preprocessing applied yet")

## 7. Preprocess NUMERICAL Features (Fit on Train, Apply to All)

In [ ]:
print("\n Preprocessing NUMERICAL Features")
print("=" * 70)

# Handle missing values in numeric features
print("\nStep 1: Handle Missing Values (using TRAIN statistics)")
numeric_medians = X_train[NUMERIC_FEATURES].median()
print("  Computed medians from TRAIN data")

# Fill missing values in all splits using train statistics
X_train[NUMERIC_FEATURES] = X_train[NUMERIC_FEATURES].fillna(numeric_medians)
X_val[NUMERIC_FEATURES] = X_val[NUMERIC_FEATURES].fillna(numeric_medians)
X_test[NUMERIC_FEATURES] = X_test[NUMERIC_FEATURES].fillna(numeric_medians)
print("   Applied to all splits")

# Scale numerical features
print("\nStep 2: Scale Numerical Features (fit on TRAIN only)")
scaler = StandardScaler()
scaler.fit(X_train[NUMERIC_FEATURES])
print("   Fitted StandardScaler on TRAIN data")

# Apply scaler to all splits
X_train[NUMERIC_FEATURES] = scaler.transform(X_train[NUMERIC_FEATURES])
X_val[NUMERIC_FEATURES] = scaler.transform(X_val[NUMERIC_FEATURES])
X_test[NUMERIC_FEATURES] = scaler.transform(X_test[NUMERIC_FEATURES])
print("   Applied to all splits")

print("\n  Numerical preprocessing complete")

## 8. Preprocess CATEGORICAL Features (One-Hot Encoding)

In [ ]:
print("\n  Preprocessing CATEGORICAL Features")
print("=" * 70)
print("Using One-Hot Encoding (fitted on TRAIN, applied to all)")

# Fit one-hot encoding on TRAIN data only
print("\nStep 1: One-Hot Encoding (fit on TRAIN only)")
X_train_cat = pd.get_dummies(
    X_train[CATEGORICAL_FEATURES], columns=CATEGORICAL_FEATURES, drop_first=False, prefix=["pos", "stage", "result"]
)
print(f"    Fit on TRAIN, created {X_train_cat.shape[1]} categorical columns")

# Get the category columns from train
cat_columns = X_train_cat.columns.tolist()
print(f"  Categories: {cat_columns}")

# Apply same encoding to val and test
print("\nStep 2: Apply same encoding to VAL and TEST")
X_val_cat = pd.get_dummies(
    X_val[CATEGORICAL_FEATURES], columns=CATEGORICAL_FEATURES, drop_first=False, prefix=["pos", "stage", "result"]
)

X_test_cat = pd.get_dummies(
    X_test[CATEGORICAL_FEATURES], columns=CATEGORICAL_FEATURES, drop_first=False, prefix=["pos", "stage", "result"]
)

# Ensure all splits have same columns (handle unseen categories)
for col in cat_columns:
    if col not in X_val_cat.columns:
        X_val_cat[col] = 0
    if col not in X_test_cat.columns:
        X_test_cat[col] = 0

X_val_cat = X_val_cat[cat_columns]
X_test_cat = X_test_cat[cat_columns]

print(f"   Applied to VAL, created {X_val_cat.shape[1]} columns")
print(f"   Applied to TEST, created {X_test_cat.shape[1]} columns")

print("\n Categorical preprocessing complete")

## 9. Combine Numerical + Categorical Features

In [ ]:
print("\n Combining Numerical + Categorical Features")

# Drop original categorical columns and add encoded ones
X_train_processed = X_train.drop(CATEGORICAL_FEATURES, axis=1).copy()
X_train_processed = pd.concat([X_train_processed.reset_index(drop=True), X_train_cat.reset_index(drop=True)], axis=1)

X_val_processed = X_val.drop(CATEGORICAL_FEATURES, axis=1).copy()
X_val_processed = pd.concat([X_val_processed.reset_index(drop=True), X_val_cat.reset_index(drop=True)], axis=1)

X_test_processed = X_test.drop(CATEGORICAL_FEATURES, axis=1).copy()
X_test_processed = pd.concat([X_test_processed.reset_index(drop=True), X_test_cat.reset_index(drop=True)], axis=1)

print("\n Combined Dataset Shapes:")
print(f"  Train: {X_train_processed.shape}")
print(f"  Val:   {X_val_processed.shape}")
print(f"  Test:  {X_test_processed.shape}")
print(f"  Features: {X_train_processed.shape[1]}")
print(f"    - Numeric (scaled): {len(NUMERIC_FEATURES)}")
print(f"    - Categorical (one-hot): {len(cat_columns)}")

print("\n Combination complete")

## 10. Add Target Variable & Create Final DataFrames

In [ ]:
print("\n Creating Final Processed Datasets")

# Add target variable
train_df = pd.concat([X_train_processed.reset_index(drop=True), y_train.reset_index(drop=True)], axis=1)

val_df = pd.concat([X_val_processed.reset_index(drop=True), y_val.reset_index(drop=True)], axis=1)

test_df = pd.concat([X_test_processed.reset_index(drop=True), y_test.reset_index(drop=True)], axis=1)

print("\n Final DataFrames:")
print(f"  Train shape: {train_df.shape}")
print(f"  Val shape:   {val_df.shape}")
print(f"  Test shape:  {test_df.shape}")
print(f"\n  Columns: {list(train_df.columns[:3])} ... [{TARGET}]")

## 11. Verify Data Quality & No Leakage

In [ ]:
print("\n Data Quality Checks")
print("=" * 70)

print("\n1. Missing Values:")
train_missing = train_df.isnull().sum().sum()
val_missing = val_df.isnull().sum().sum()
test_missing = test_df.isnull().sum().sum()
print(f"  Train: {train_missing} missing values")
print(f"  Val:   {val_missing} missing values")
print(f"  Test:  {test_missing} missing values")

print("\n2. Target Distribution:")
print(f"  Train - Mean: {train_df[TARGET].mean():.3f}, Std: {train_df[TARGET].std():.3f}")
print(f"  Val   - Mean: {val_df[TARGET].mean():.3f}, Std: {val_df[TARGET].std():.3f}")
print(f"  Test  - Mean: {test_df[TARGET].mean():.3f}, Std: {test_df[TARGET].std():.3f}")

print("\n3. Feature Statistics (Numeric Features):")
print(
    f"  Train - Mean: {X_train_processed[NUMERIC_FEATURES].values.mean():.4f}, Std: {X_train_processed[NUMERIC_FEATURES].values.std():.4f}"
)
print(
    f"  Val   - Mean: {X_val_processed[NUMERIC_FEATURES].values.mean():.4f}, Std: {X_val_processed[NUMERIC_FEATURES].values.std():.4f}"
)
print(
    f"  Test  - Mean: {X_test_processed[NUMERIC_FEATURES].values.mean():.4f}, Std: {X_test_processed[NUMERIC_FEATURES].values.std():.4f}"
)
print("  (Note: Train should be ~0 mean and ~1 std after scaling)")

print("\n4. No Data Leakage Check:")
print("   Splits are disjoint")
print("   Scaler fitted on train only")
print("   Categorical encoder fitted on train only")
print("   No val/test data used in fitting")

print("\n All quality checks passed!")

## 12. Visualize Data Quality

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("Processed Data Quality Check", fontsize=14, fontweight="bold")

# Target distribution
axes[0, 0].hist(train_df[TARGET], bins=20, alpha=0.7, label="Train", color="blue")
axes[0, 0].hist(val_df[TARGET], bins=20, alpha=0.7, label="Val", color="green")
axes[0, 0].hist(test_df[TARGET], bins=20, alpha=0.7, label="Test", color="red")
axes[0, 0].set_title("Target Distribution (player_rating)")
axes[0, 0].set_xlabel("Rating")
axes[0, 0].set_ylabel("Frequency")
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Sample sizes
sizes = [len(train_df), len(val_df), len(test_df)]
labels = ["Train", "Val", "Test"]
colors = ["blue", "green", "red"]
axes[0, 1].bar(labels, sizes, color=colors, alpha=0.7)
axes[0, 1].set_title("Split Sizes")
axes[0, 1].set_ylabel("Number of Samples")
for i, size in enumerate(sizes):
    axes[0, 1].text(i, size, f"{size:,}", ha="center", va="bottom")
axes[0, 1].grid(True, alpha=0.3, axis="y")

# Numeric feature distribution (after scaling)
axes[1, 0].hist(X_train_processed[NUMERIC_FEATURES].values.flatten(), bins=30, alpha=0.7, label="Train", color="blue")
axes[1, 0].axvline(0, color="red", linestyle="--", linewidth=2, label="Mean=0")
axes[1, 0].set_title("Numeric Features Distribution (Scaled)")
axes[1, 0].set_xlabel("Scaled Value")
axes[1, 0].set_ylabel("Frequency")
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# Summary statistics table
stats_text = f"""
PREPROCESSING SUMMARY

Numeric Features: {len(NUMERIC_FEATURES)}
Categorical Features: {len(CATEGORICAL_FEATURES)}
Total Features: {train_df.shape[1] - 1}

Train: {len(train_df):,} samples
Val:   {len(val_df):,} samples
Test:  {len(test_df):,} samples

Preprocessing Applied:
 Missing value imputation
 Numeric scaling (StandardScaler)
 Categorical encoding (One-Hot)
 NO data leakage
"""
axes[1, 1].text(
    0.1, 0.5, stats_text, fontsize=11, family="monospace", verticalalignment="center", transform=axes[1, 1].transAxes
)
axes[1, 1].axis("off")

plt.tight_layout()
plt.show()

## 13. Save Processed Datasets

In [ ]:
print("\n Saving Processed Datasets")
print("=" * 70)

# Define output paths
train_path = os.path.join(OUTPUT_DIR, "train.csv")
val_path = os.path.join(OUTPUT_DIR, "validation.csv")
test_path = os.path.join(OUTPUT_DIR, "test.csv")

# Save CSV files
train_df.to_csv(train_path, index=False)
val_df.to_csv(val_path, index=False)
test_df.to_csv(test_path, index=False)

print(f"\n Train: {train_path}")
print(f"   Shape: {train_df.shape}")
print(f"   Size: {os.path.getsize(train_path) / 1024**2:.2f} MB")

print(f"\n Validation: {val_path}")
print(f"   Shape: {val_df.shape}")
print(f"   Size: {os.path.getsize(val_path) / 1024**2:.2f} MB")

print(f"\n Test: {test_path}")
print(f"   Shape: {test_df.shape}")
print(f"   Size: {os.path.getsize(test_path) / 1024**2:.2f} MB")

In [ ]:
# Save preprocessing metadata for reference
print("\n Saving Metadata")

metadata = {
    "numeric_features": NUMERIC_FEATURES,
    "categorical_features": CATEGORICAL_FEATURES,
    "categorical_columns_onehot": cat_columns,
    "engineered_features": ENGINEERED_FEATURES,
    "target_variable": TARGET,
    "total_features": train_df.shape[1] - 1,
    "train_samples": len(train_df),
    "val_samples": len(val_df),
    "test_samples": len(test_df),
    "preprocessing_steps": [
        "Removed zero ratings",
        "Created engineered features",
        "Split data (70/15/15)",
        "Filled missing values (train median)",
        "Scaled numeric features (train fitted)",
        "One-hot encoded categorical features (train fitted)",
        "Combined numeric + categorical",
    ],
}

import json

metadata_path = os.path.join(OUTPUT_DIR, "preprocessing_metadata.json")
with open(metadata_path, "w") as f:
    json.dump(metadata, f, indent=2)

print(f" Metadata: {metadata_path}")

In [ ]:
# Save scaler for future use
print("\n Saving Transformers")

scaler_path = os.path.join(OUTPUT_DIR, "scaler.pkl")
with open(scaler_path, "wb") as f:
    pickle.dump(scaler, f)
print(f" Scaler: {scaler_path}")

# Save categorical mapping for reference
cat_mapping = {
    "original_categories": CATEGORICAL_FEATURES,
    "onehot_columns": cat_columns,
    "encoding_method": "One-Hot (fitted on train)",
}

cat_mapping_path = os.path.join(OUTPUT_DIR, "categorical_mapping.json")
with open(cat_mapping_path, "w") as f:
    json.dump(cat_mapping, f, indent=2)
print(f" Categorical Mapping: {cat_mapping_path}")

## 14. Create Summary Report

In [ ]:
summary = f"""
DATA PREPROCESSING PIPELINE - SUMMARY REPORT
{"=" * 70}

OBJECTIVE:
  Extract, preprocess, and split player performance data
  WITHOUT data leakage

DATA SOURCE:
  Original dataset: {INPUT_CSV}
  Original shape: (54600, 75)
  After filtering: ({len(df):,}, {df.shape[1]})

FEATURES SELECTED:
  Numeric Features: {len(NUMERIC_FEATURES)}
    - Consistency score, market value, pressure resistance
    - Defensive/offensive contribution, creativity, stamina
    - Age, height, weight, minutes played
    - Goals, tackles, interceptions, key passes
    - Top speed, distance covered, accelerations, decelerations
    - Engineered: pass accuracy, shot accuracy, dribble success, etc.

  Categorical Features: {len(CATEGORICAL_FEATURES)}
    - Position (4 values: GK, DF, MF, FW)
    - Tournament Stage (7 values: Group Stage → Final)
    - Match Result (3 values: W, D, L)

TARGET VARIABLE:
  Variable: {TARGET}
  Original range: 0.00 - 9.40
  Mean: {df[TARGET].mean():.2f}
  Std: {df[TARGET].std():.2f}

PREPROCESSING STEPS (IN ORDER):
  1. Removed zero ratings (non-playing appearances)
  2. Created engineered features (5 new numeric features)
  3. SPLIT DATA (70/15/15) ← BEFORE any transformations
  4. Imputed missing values using TRAIN statistics only
  5. Scaled numeric features using StandardScaler fitted on TRAIN
  6. One-hot encoded categorical features using TRAIN categories
  7. Applied transformations to VAL and TEST using TRAIN-fitted transformers
  8. Combined numeric + categorical features

DATA LEAKAGE PREVENTION:
    Split BEFORE preprocessing
    Fitted scaler only on TRAIN data
    Fitted encoder only on TRAIN data
    Applied same transformations to VAL/TEST
    No validation/test statistics used in preprocessing

FINAL DATASETS:
  Train: {len(train_df):6,} samples ({len(train_df) / len(df) * 100:5.1f}%)
  Val:   {len(val_df):6,} samples ({len(val_df) / len(df) * 100:5.1f}%)
  Test:  {len(test_df):6,} samples ({len(test_df) / len(df) * 100:5.1f}%)
  
  Features: {train_df.shape[1] - 1}
    - Numeric (scaled): {len(NUMERIC_FEATURES)}
    - Categorical (one-hot): {len(cat_columns)}
  
  Target: {TARGET}
  Train mean: {train_df[TARGET].mean():.3f}
  Val mean:   {val_df[TARGET].mean():.3f}
  Test mean:  {test_df[TARGET].mean():.3f}

OUTPUT FILES:
  Location: {OUTPUT_DIR}
  
  1. train.csv
     - {len(train_df):,} rows × {train_df.shape[1]} columns
     - Contains: all features + target variable
     - Ready for: model training
  
  2. validation.csv
     - {len(val_df):,} rows × {val_df.shape[1]} columns
     - Contains: all features + target variable
     - Ready for: hyperparameter tuning
  
  3. test.csv
     - {len(test_df):,} rows × {test_df.shape[1]} columns
     - Contains: all features + target variable
     - Ready for: final model evaluation
  
  4. preprocessing_metadata.json
     - Feature lists and configuration
  
  5. scaler.pkl
     - StandardScaler fitted on TRAIN data
     - Use for scaling new data in production
  
  6. categorical_mapping.json
     - One-hot encoded column names


QUALITY ASSURANCE:
   No missing values in processed data
   No data leakage between splits
   Features properly scaled
   Categorical features properly encoded
   Target distribution similar across splits
   All splits have same feature structure

{"=" * 70}
Generated by: Feature Extraction & Preprocessing Pipeline
Encoding Method: One-Hot (fitted on TRAIN only)
No data leakage: VERIFIED
"""

summary_path = os.path.join(OUTPUT_DIR, "PREPROCESSING_SUMMARY.txt")
with open(summary_path, "w") as f:
    f.write(summary)

print(summary)
print(f"\n Summary saved to: {summary_path}")

## 15. Final Verification & Load Test

In [ ]:
print("\n FINAL VERIFICATION")
print("=" * 70)

# Verify files exist
print("\nChecking output files:")
for filename in [
    "train.csv",
    "validation.csv",
    "test.csv",
    "preprocessing_metadata.json",
    "scaler.pkl",
    "categorical_mapping.json",
    "PREPROCESSING_SUMMARY.txt",
]:
    filepath = os.path.join(OUTPUT_DIR, filename)
    if os.path.exists(filepath):
        size = os.path.getsize(filepath)
        if filename.endswith(".csv"):
            size_str = f"{size / 1024**2:.2f} MB"
        else:
            size_str = f"{size / 1024:.1f} KB"
        print(f"   {filename:40} ({size_str})")
    else:
        print(f"   {filename:40} (NOT FOUND)")

In [ ]:
# Load test - verify data can be read
print("\nLoad test (verifying data integrity):")

train_test = pd.read_csv(train_path)
val_test = pd.read_csv(val_path)
test_test = pd.read_csv(test_path)

print(f"   Train loaded: {train_test.shape}")
print(f"   Val loaded:   {val_test.shape}")
print(f"   Test loaded:  {test_test.shape}")

print("\nTarget variable check:")
print(f"  Train {TARGET}: {train_test[TARGET].notna().sum()} / {len(train_test)} non-null")
print(f"  Val {TARGET}:   {val_test[TARGET].notna().sum()} / {len(val_test)} non-null")
print(f"  Test {TARGET}:  {test_test[TARGET].notna().sum()} / {len(test_test)} non-null")

print("\n DATA PREPROCESSING COMPLETE!")
print(f"\n Output directory: {OUTPUT_DIR}")
print("\n Ready for modeling!")